# GBD_9 — 2-D single-beamlet propagator (Era 2, step 1)

This notebook is the 2-D analog of `GBD_5`. We take **one** Gaussian beamlet, propagate it analytically with the complex q-parameter form, and verify it against the textbook 2-D Gaussian beam closed-forms. No basis sum yet (that is `GBD_10`); no FFT reference yet (that is `GBD_11`).

**Why this is the right first step into Era 2.** Every later 2-D notebook will rest on the per-beamlet propagator. If it is wrong here, every basis-sum diagnostic in `GBD_10` and every FFT comparison in `GBD_11` would be polluted by it and the bug would be hard to localise. So we exercise it in isolation, on one beamlet, against analytic predictions, before stacking anything on top.

**What changes vs. 1-D (subtle but important).**

The 2-D Gaussian beam is *almost* a tensor product of two 1-D beams, but two factors are different and easy to get wrong:

1. **Amplitude scaling.** 1-D is `sqrt(w0/w(z))`. 2-D is `w0/w(z)` (no square root) — because the x-Gaussian and y-Gaussian each carry a `sqrt(w0/w(z))` factor, so they multiply. In the q-parameter form this means the prefactor is `q0/q(z)` (not `sqrt(q0/q(z))`).
2. **Gouy phase.** 1-D is `(1/2)·arctan(z/zR)`. 2-D is the full `arctan(z/zR)` — twice as much. Same root cause: it's `arg(q0/q(z))` rather than `arg(sqrt(q0/q(z)))`.

Everything else carries over: `w(z) = w0·sqrt(1+(z/zR)²)`, `R(z) = z·(1+(zR/z)²)`, `zR = π·w0²/λ`, outgoing-wave convention `exp(+ikz)` so `q0 = -i·zR`.

**LiDAR parameters are carried forward from GBD_8.** λ = 1550 nm, w₀ = 1 mm. So zR ≈ 2.03 m and we work in physical units (mm for x, y; m for z).

**Sanity checks performed below.**
1. z = 0 identity — propagator at z = 0 must equal a directly-evaluated 2-D Gaussian to machine precision.
2. Multi-z table — `w(z)`, `R(z)`, Gouy phase extracted from the propagated field must match textbook formulas.
3. 1-D-slice consistency — a y = 0 slice with `ky = 0`, modulo the known amplitude-scaling difference, must match an independent 1-D propagation.
4. Tilt walk — centroid moves to `(kx/k)·z, (ky/k)·z` for a tilted beam.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- LiDAR-realistic parameters carried forward from GBD_8 ---
wavelength = 1550e-9        # 1550 nm, in meters
w0 = 1.0e-3                 # 1 mm beam waist, in meters
k = 2*np.pi / wavelength    # wavenumber
zR = np.pi * w0**2 / wavelength   # Rayleigh range

# Paraxial validity diagnostic (cf. CONCEPTS.md Topic 2 — we run this every notebook)
theta_div = wavelength / (np.pi * w0)
print(f'wavelength = {wavelength*1e9:.1f} nm')
print(f'w0        = {w0*1e3:.3f} mm')
print(f'zR        = {zR:.4f} m')
print(f'theta_div = {theta_div:.3e} rad   (paraxial OK if << 1; here ~ 5e-4)')

# --- 2-D grid ---
# At z = 5*zR the beam spreads to w0*sqrt(26) ~ 5.1 mm. ±20 mm grid keeps it well-contained.
Lx = 20e-3       # half-extent in x, meters
Ly = 20e-3
Nx = 257         # odd so the center sample sits exactly on (0,0); this matters only for the diagnostics below
Ny = 257
x = np.linspace(-Lx, Lx, Nx)
y = np.linspace(-Ly, Ly, Ny)
X, Y = np.meshgrid(x, y, indexing='xy')
print(f'\nGrid: {Nx} x {Ny} points, x in [{-Lx*1e3:.1f}, {Lx*1e3:.1f}] mm, y in [{-Ly*1e3:.1f}, {Ly*1e3:.1f}] mm')

## The 2-D propagator

Direct generalisation of `GBD_5`'s 1-D propagator. Three lines actually change:
- `(q0/qz)` instead of `sqrt(q0/qz)` (amplitude/Gouy);
- transverse argument now `(X-x_c)² + (Y-y_c)²`;
- `kz = sqrt(k² - kx² - ky²)` and tilt phase `exp(i(kx·X + ky·Y))`.

Returns the complex field `E(x, y)` at distance z, plus a small diagnostics dict for the verification step.

In [ ]:
def propagate_tilted_gaussian_2d(X, Y, x0, y0, w0, kx, ky, k, z):
    """Propagate a 2-D tilted Gaussian beamlet by the analytic q-parameter form.

    Inputs:
        X, Y    : 2-D meshgrid arrays (Ny, Nx) of transverse coordinates [m]
        x0, y0  : launch waist position [m]
        w0      : waist radius (1/e^2 of intensity) [m]
        kx, ky  : transverse wavevectors [1/m]; kx=ky=0 for an on-axis beam
        k       : total wavevector magnitude [1/m]
        z       : propagation distance [m]
    Returns:
        E       : complex field array of shape (Ny, Nx)
        diag    : dict of analytic w(z), R(z), Gouy(z), x_c, y_c, kz
    """
    wl = 2*np.pi / k
    zR_loc = np.pi * w0**2 / wl

    # walked center for tilted beams
    x_c = x0 + (kx/k) * z
    y_c = y0 + (ky/k) * z

    # complex q parameter
    q0 = -1j * zR_loc
    qz = z + q0

    # 2-D transverse field — note (q0/qz) NOT sqrt(q0/qz). 2-D = product of two 1-D factors.
    transverse = (q0/qz) * np.exp(1j * k * ((X-x_c)**2 + (Y-y_c)**2) / (2*qz))

    # longitudinal phase (kz = sqrt(k^2 - kx^2 - ky^2) for outgoing wave)
    kz = np.sqrt(k**2 - kx**2 - ky**2)
    longitudinal = np.exp(1j * kz * z)

    # transverse tilt phase (carries the angle information)
    tilt = np.exp(1j * (kx*X + ky*Y))

    E = transverse * longitudinal * tilt

    # textbook closed-form diagnostics for verification
    if z == 0:
        w_z = w0
        R_z = np.inf
        gouy = 0.0
    else:
        w_z = w0 * np.sqrt(1 + (z/zR_loc)**2)
        R_z = z * (1 + (zR_loc/z)**2)
        gouy = -np.arctan(z/zR_loc)        # full (2-D) Gouy; sign is - because q0 = -i*zR (outgoing-wave convention)
    diag = dict(w=w_z, R=R_z, gouy=gouy, x_c=x_c, y_c=y_c, kz=kz, zR=zR_loc)
    return E, diag

## Sanity 1: z = 0 identity

A direct evaluation of the 2-D Gaussian at z = 0 must equal the propagator output at z = 0. Both should be a clean `exp(-(x²+y²)/w0²)` (intensity), with the propagator's prefactor reducing to `q0/q0 = 1`. Difference is machine epsilon.

In [ ]:
# Direct evaluation of a 2-D Gaussian at z = 0
E_direct = np.exp(-(X**2 + Y**2) / w0**2)

# Propagator at z = 0
E_prop_z0, _ = propagate_tilted_gaussian_2d(X, Y, x0=0, y0=0, w0=w0, kx=0, ky=0, k=k, z=0.0)

print(f'Max |E_prop(z=0) - E_direct| = {np.max(np.abs(E_prop_z0 - E_direct)):.3e}')
print('(should be ~1e-15, machine epsilon)')

## Sanity 2: closed-form match at multiple z

Propagate to z = 0, zR/2, zR, 2·zR, 5·zR and extract `w(z)` and on-axis Gouy phase from the *propagated field*; compare to the textbook formulas. Wavefront curvature `R(z)` is checked by looking at the quadratic phase coefficient on-axis.

In [ ]:
def measure_w_from_intensity(E, x, y):
    """Measure 1/e^2 intensity radius assuming axisymmetric Gaussian centered at origin."""
    I = np.abs(E)**2
    # second moment — exact for a 2-D Gaussian, robust to numerical noise
    X_, Y_ = np.meshgrid(x, y, indexing='xy')
    total = I.sum()
    sigma2_x = (I * X_**2).sum() / total
    sigma2_y = (I * Y_**2).sum() / total
    # for a Gaussian intensity exp(-2 r^2 / w^2), <r^2>/2 = w^2/4 => w = 2*sqrt(<x^2>)
    return 2.0 * np.sqrt(0.5 * (sigma2_x + sigma2_y))

def measure_gouy_on_axis(E, x, y):
    """On-axis phase, modulo the longitudinal kz*z (which we'll subtract analytically)."""
    ix = len(x)//2
    iy = len(y)//2
    return np.angle(E[iy, ix])

def measure_R_from_phase(E, x, y, k_local, w_z):
    """Fit the quadratic transverse phase coefficient to recover R(z).
    The phase along x at y=0 is k*(x^2)/(2R) plus a constant on-axis offset.
    Pick a small window around the axis (within w(z)) to avoid 2π wraps."""
    iy0 = len(y)//2
    phase = np.unwrap(np.angle(E[iy0, :]))
    mask = np.abs(x) < 0.5*w_z   # stay well inside the beam to avoid noise
    # fit phase(x) = a*x^2 + b*x + c; recover R from a = k/(2R)
    coeffs = np.polyfit(x[mask], phase[mask], 2)
    a = coeffs[0]
    if abs(a) < 1e-30:
        return np.inf
    return k_local / (2*a)

z_values = np.array([0.0, zR/2, zR, 2*zR, 5*zR])
rows = []
for z_test in z_values:
    E_z, diag = propagate_tilted_gaussian_2d(X, Y, 0, 0, w0, 0, 0, k, z_test)
    w_meas = measure_w_from_intensity(E_z, x, y)
    if z_test == 0:
        R_meas = np.inf
        gouy_meas = 0.0
    else:
        # subtract the analytic kz*z so what's left on-axis is the Gouy term
        gouy_meas = np.angle(E_z[Ny//2, Nx//2] * np.exp(-1j*diag['kz']*z_test))
        R_meas = measure_R_from_phase(E_z * np.exp(-1j*diag['kz']*z_test), x, y, k, diag['w'])
    rows.append((z_test, w_meas, diag['w'], R_meas, diag['R'], gouy_meas, diag['gouy']))

print(f"{'z [m]':>8s}  {'w_meas [mm]':>12s}  {'w_text [mm]':>12s}  {'R_meas [m]':>12s}  {'R_text [m]':>12s}  {'Gouy_meas':>10s}  {'Gouy_text':>10s}")
for z_test, w_m, w_t, R_m, R_t, g_m, g_t in rows:
    R_m_str = 'inf' if not np.isfinite(R_m) else f'{R_m:.4f}'
    R_t_str = 'inf' if not np.isfinite(R_t) else f'{R_t:.4f}'
    print(f'{z_test:8.4f}  {w_m*1e3:12.5f}  {w_t*1e3:12.5f}  {R_m_str:>12s}  {R_t_str:>12s}  {g_m:10.5f}  {g_t:10.5f}')

## Multi-z plot — amplitude and phase

Top row: |E| at each z. Bottom row: arg(E) with the longitudinal `kz·z` subtracted, so what we see is the *envelope phase* — wavefront curvature plus Gouy. This is the FMCW-relevant phase that the receiver will see.

In [ ]:
fig, axes = plt.subplots(2, len(z_values), figsize=(4*len(z_values), 7))
for i, z_test in enumerate(z_values):
    E_z, diag = propagate_tilted_gaussian_2d(X, Y, 0, 0, w0, 0, 0, k, z_test)
    # subtract longitudinal phase so amplitude/curvature/Gouy are visible
    E_env = E_z * np.exp(-1j*diag['kz']*z_test)
    extent = [-Lx*1e3, Lx*1e3, -Ly*1e3, Ly*1e3]

    ax = axes[0, i]
    im = ax.imshow(np.abs(E_env), origin='lower', extent=extent, cmap='viridis')
    ax.set_title(f'|E|, z = {z_test:.3f} m\nw(z) = {diag["w"]*1e3:.3f} mm')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    plt.colorbar(im, ax=ax, fraction=0.046)

    ax = axes[1, i]
    im = ax.imshow(np.angle(E_env), origin='lower', extent=extent, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    ax.set_title(f'arg(E), z = {z_test:.3f} m\nGouy = {diag["gouy"]:.3f} rad')
    ax.set_xlabel('x [mm]'); ax.set_ylabel('y [mm]')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Sanity 3: 1-D slice consistency with GBD_5/6/8

Take a y = 0 slice with `ky = 0`. The transverse term is then `(q0/qz)·exp(i·k·x²/(2qz))`. The 1-D propagator from GBD_5 has `sqrt(q0/qz)·exp(i·k·x²/(2qz))`. So they differ only by the prefactor: 2-D / 1-D = `sqrt(q0/qz)`.

The check below evaluates the 1-D propagator independently at the same parameters and compares spatial structures. After dividing out the known prefactor difference, the residual should be machine epsilon.

In [ ]:
def propagate_tilted_gaussian_1d(x_arr, x0, w0, kx, k, z):
    """1-D version (carried over from GBD_5)."""
    wl = 2*np.pi / k
    zR_loc = np.pi * w0**2 / wl
    x_c = x0 + (kx/k) * z
    q0 = -1j * zR_loc
    qz = z + q0
    envelope = np.sqrt(q0/qz) * np.exp(1j * k * (x_arr - x_c)**2 / (2*qz))
    kz = np.sqrt(k**2 - kx**2)
    return envelope * np.exp(1j*kz*z) * np.exp(1j*kx*x_arr)

z_test = zR
E_2d, diag = propagate_tilted_gaussian_2d(X, Y, 0, 0, w0, 0, 0, k, z_test)
slice_2d = E_2d[Ny//2, :]   # y = 0 slice
slice_1d = propagate_tilted_gaussian_1d(x, 0, w0, 0, k, z_test)

# Predicted ratio: 2-D / 1-D = sqrt(q0/qz)
q0 = -1j*zR
qz = z_test + q0
ratio_pred = np.sqrt(q0/qz)

ratio_meas = slice_2d / slice_1d
# pick a point well within the beam to avoid 0/0
idx_center = Nx//2
print(f'2-D/1-D at center: {ratio_meas[idx_center]:.6e}')
print(f'Predicted (sqrt(q0/qz)): {ratio_pred:.6e}')
print(f'Difference: {np.abs(ratio_meas[idx_center] - ratio_pred):.3e}  (should be ~1e-15)')

# After removing the prefactor, spatial structure must match exactly
residual = slice_2d/ratio_pred - slice_1d
print(f'\nMax |slice_2d/sqrt(q0/qz) - slice_1d| = {np.max(np.abs(residual)):.3e}')
print('(should be ~1e-15: 1-D propagator is recovered exactly from a 2-D y=0 slice modulo the known prefactor)')

## Sanity 4: tilt walk

With `kx = k·sin(0.001)` (1 mrad — appropriate for LiDAR scan angles) and `ky = k·sin(0.0005)`, the centroid should walk to `(kx/k)·z, (ky/k)·z`. We pick a moderate z = zR and verify by measuring the intensity centroid.

In [ ]:
theta_x = 1e-3      # rad
theta_y = 5e-4      # rad
kx_t = k * np.sin(theta_x)
ky_t = k * np.sin(theta_y)

z_test = zR
E_t, diag = propagate_tilted_gaussian_2d(X, Y, 0, 0, w0, kx_t, ky_t, k, z_test)
I = np.abs(E_t)**2
x_centroid = (I * X).sum() / I.sum()
y_centroid = (I * Y).sum() / I.sum()

expected_x = (kx_t/k) * z_test
expected_y = (ky_t/k) * z_test
print(f'Measured  centroid: ({x_centroid*1e3:+.6f}, {y_centroid*1e3:+.6f}) mm')
print(f'Expected  centroid: ({expected_x*1e3:+.6f}, {expected_y*1e3:+.6f}) mm')
print(f'Diag x_c:           {diag["x_c"]*1e3:+.6f} mm')
print(f'Diag y_c:           {diag["y_c"]*1e3:+.6f} mm')
print(f'\nResiduals: dx = {abs(x_centroid-expected_x):.3e} m, dy = {abs(y_centroid-expected_y):.3e} m')
print('(should be tiny — limited only by grid resolution and that we are looking at a Gaussian intensity centroid)')

## Summary

If all four sanity checks pass:
- Sanity 1: machine-precision identity at z = 0.
- Sanity 2: measured `w(z)`, `R(z)`, Gouy phase agree with textbook formulas.
- Sanity 3: y = 0 slice of 2-D propagator equals 1-D propagator times the known `sqrt(q0/qz)` factor — confirms tensor-product structure.
- Sanity 4: tilted-beam centroid walks to `(kx/k)·z, (ky/k)·z`.

...then the 2-D per-beamlet propagator is verified, and `GBD_10` can build the basis sum on top of it.

**What this notebook deliberately did not do** (saving for later):
- Sum a basis of multiple 2-D beamlets — `GBD_10`.
- Compare against an independent 2-D angular-spectrum FFT — `GBD_11`.
- Sweep z over the LiDAR operating range — `GBD_11`.
- Stress-test on non-Gaussian sources — `GBD_12` (conditional).